In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip -q install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.7 MB/s eta 0:00:00


In [ ]:
# ===============================================
# Build a NEW, CLEAN VKB from a data split (no prior VKB)
# Pipeline (no optional steps):
#   1) List images by split/scene
#   2) Detect objects (YOLO) → per-image {obj: max_conf}
#   3) Per scene: image_count, freq_pct, mean_conf (trimmed)
#   4) Across scenes: tau(s), df(o), idf(o)
#   5) score(o,s) = freq_pct * mean_conf * idf
#   6) key_objects: smallest prefix reaching 90% coverage
# ===============================================

# If running in Colab, uncomment:
# !pip install ultralytics --quiet

import os
import json
import math
import gc
from typing import Dict, List, Tuple, Any
from collections import defaultdict

import numpy as np
import torch
from ultralytics import YOLO

# ----------------------------
# CONFIG — edit these paths only
# ----------------------------
DATA_ROOT = "/content/drive/MyDrive/Scene_Classification/final_Split2"  # expects DATA_ROOT/<split>/<scene>/*.jpg
SPLIT = "train"  # which split to build the VKB from
OUTPUT_CLEAN_VKB_PATH = "/content/drive/MyDrive/Scene_Classification/clean_vkb_new_from_split.json"

# YOLO runtime (safe defaults)
MODEL_WEIGHTS = "yolov8l.pt"
IMG_SIZE = 640
CONF_THRES = 0.25
IOU_THRES = 0.6
MAX_DET = 100
INIT_BATCH = 32
MIN_BATCH = 4
DEVICE = 0 if torch.cuda.is_available() else "cpu"
USE_HALF = torch.cuda.is_available()  # half precision only on CUDA
SUPPORTED_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

COVERAGE_TARGET = 0.90  # 90% coverage for key selection (fixed)

torch.backends.cudnn.benchmark = True  # speed knob


# ----------------------------
# Helpers: IO
# ----------------------------
def save_json(path: str, data: Any) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)


# ----------------------------
# Step 1: list images by split/scene
# ----------------------------
def list_scenes_and_images(data_root: str, split: str) -> Dict[str, List[str]]:
    split_dir = os.path.join(data_root, split)
    if not os.path.isdir(split_dir):
        raise RuntimeError(f"Split directory not found: {split_dir}")
    out: Dict[str, List[str]] = {}
    for scene in sorted(os.listdir(split_dir)):
        scene_dir = os.path.join(split_dir, scene)
        if not os.path.isdir(scene_dir):
            continue
        imgs = [
            os.path.join(scene_dir, fn)
            for fn in os.listdir(scene_dir)
            if fn.lower().endswith(SUPPORTED_EXTS)
        ]
        imgs.sort()
        if imgs:
            out[scene] = imgs
    if not out:
        raise RuntimeError(f"No scenes/images under: {split_dir}")
    return out


# ----------------------------
# Step 2: YOLO detection (unique-per-image max conf)
# ----------------------------
def predict_paths(model: YOLO, paths: List[str], batch_size: int) -> Dict[str, Dict[str, float]]:
    """
    Run YOLO on image paths and return per-image unique detections:
      {image_path: {class_name: max_conf}}
    Includes dynamic batch backoff on CUDA OOM.
    """
    out: Dict[str, Dict[str, float]] = {}
    bs = batch_size
    i = 0
    while i < len(paths):
        chunk = paths[i : i + bs]
        try:
            with torch.inference_mode():
                results = model.predict(
                    source=chunk,
                    imgsz=IMG_SIZE,
                    conf=CONF_THRES,
                    iou=IOU_THRES,
                    max_det=MAX_DET,
                    device=DEVICE,
                    half=USE_HALF,
                    verbose=False,
                )
            for p, r in zip(chunk, results):
                per_image: Dict[str, float] = {}
                if getattr(r, "boxes", None) is not None and len(r.boxes) > 0:
                    names_map = r.names if getattr(r, "names", None) else model.names
                    for b in r.boxes:
                        cls_id = int(b.cls.item())
                        name = names_map[cls_id]
                        conf = float(b.conf.item())
                        # keep max confidence per class for this image
                        if (name not in per_image) or (conf > per_image[name]):
                            per_image[name] = conf
                out[p] = per_image
            i += bs
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            gc.collect()
            if bs <= MIN_BATCH:
                raise
            bs = max(MIN_BATCH, bs // 2)
        except Exception as e:
            print(f"⚠️ Error on batch starting at index {i}: {e}")
            # fallback: process first item, requeue others
            if len(chunk) > 1:
                for q in reversed(chunk[1:]):
                    paths.insert(i + 1, q)
                chunk = [chunk[0]]
                bs = max(MIN_BATCH, 1)
            else:
                out[chunk[0]] = {}
                i += 1
        finally:
            torch.cuda.empty_cache()
    return out


# ----------------------------
# Step 3: aggregate per scene
#   image_count, freq_pct, mean_conf (trimmed)
# ----------------------------
def trimmed_mean(values: List[float], trim_per_tail: float) -> float:
    if not values:
        return 0.0
    n = len(values)
    if n == 1 or trim_per_tail <= 0:
        return float(sum(values) / n)
    k = int(math.floor(n * trim_per_tail))
    if k == 0 or 2 * k >= n:
        return float(sum(values) / n)
    vals = sorted(values)
    trimmed = vals[k : n - k]
    if not trimmed:
        return float(sum(values) / n)
    return float(sum(trimmed) / len(trimmed))


def per_scene_unique_max(image_paths: List[str], det_map: Dict[str, Dict[str, float]]) -> List[Dict[str, float]]:
    """
    Build a list of per-image dicts {obj: max_conf}, aligned to image_paths order.
    """
    out = []
    for p in image_paths:
        out.append(det_map.get(p, {}))
    return out


def aggregate_scene_stats(scene2imgs: Dict[str, List[str]], det_map: Dict[str, Dict[str, float]]
                          ) -> Tuple[Dict[str, Any], List[str]]:
    """
    Returns:
      scene_stats: { scene: {
          "num_images": N_s,
          "trim_per_tail": t_s,
          "objects": {
            obj: {
              "image_count": int,
              "freq_pct": float,
              "mean_conf": float
            }, ...
          }
      }}
      all_objects: list of all object names seen anywhere
    """
    scene_stats: Dict[str, Any] = {}
    obj_global: Dict[str, int] = {}

    for scene, paths in scene2imgs.items():
        per_image = per_scene_unique_max(paths, det_map)
        N = max(1, len(per_image))
        trim_per_tail = min(0.10, 10.0 / N)

        conf_lists: Dict[str, List[float]] = defaultdict(list)
        for img_dict in per_image:
            for obj, conf in img_dict.items():
                conf_lists[obj].append(float(conf))
                obj_global[obj] = 1

        objects: Dict[str, Any] = {}
        for obj, confs in conf_lists.items():
            image_count = len(confs)  # one value per image where present
            freq_pct = image_count / float(N)
            mean_conf = trimmed_mean(confs, trim_per_tail)
            objects[obj] = {
                "image_count": int(image_count),
                "freq_pct": float(freq_pct),
                "mean_conf": float(mean_conf),
            }

        scene_stats[scene] = {
            "num_images": int(N),
            "trim_per_tail": float(trim_per_tail),
            "objects": objects,
        }

    all_objects = sorted(obj_global.keys())
    return scene_stats, all_objects


# ----------------------------
# Step 4: global distinctiveness (tau, df, idf)
# ----------------------------
def compute_tau_by_scene(scene_stats: Dict[str, Any]) -> Dict[str, int]:
    """tau(s) = max(3, ceil(0.02 * N_s)) per scene s."""
    tau_by_scene: Dict[str, int] = {}
    for scene, sc in scene_stats.items():
        N = int(sc["num_images"])
        tau_by_scene[scene] = max(3, math.ceil(0.02 * N))
    return tau_by_scene


def compute_idf(scene_stats: Dict[str, Any], obj_names: List[str], tau_by_scene: Dict[str, int]) -> Dict[str, float]:
    """
    df(o)  = number of scenes with image_count(o,s) >= tau(s)
    idf(o) = log((S+1)/(df+1)) + 1
    """
    scenes = list(scene_stats.keys())
    S = len(scenes)
    df = {o: 0 for o in obj_names}
    for scene in scenes:
        tau = tau_by_scene[scene]
        obj_map = scene_stats[scene]["objects"]
        for o in obj_names:
            st = obj_map.get(o)
            if st and int(st["image_count"]) >= tau:
                df[o] += 1
    idf = {o: math.log((S + 1) / (df[o] + 1)) + 1.0 for o in obj_names}
    return idf


# ----------------------------
# Step 5 & 6: score and coverage-based key selection
# ----------------------------
def build_clean_vkb(scene_stats: Dict[str, Any], idf: Dict[str, float], coverage_target: float = COVERAGE_TARGET) -> Dict[str, Any]:
    """
    score(o,s) = freq_pct * mean_conf * idf(o)
    key_objects = smallest prefix with cumulative >= coverage_target * total_score
    Only objects with image_count >= tau(s) are eligible.
    """
    vkb: Dict[str, Any] = {}
    tau_by_scene = compute_tau_by_scene(scene_stats)

    for scene, sc in scene_stats.items():
        N = int(sc["num_images"])
        t_tail = float(sc["trim_per_tail"])
        tau = tau_by_scene[scene]
        objects_in = sc["objects"]

        scored: List[Tuple[str, float]] = []
        obj_rows: Dict[str, Any] = {}
        total_score = 0.0

        # score eligible objects; others get score=0 for transparency
        for obj, st in objects_in.items():
            image_count = int(st["image_count"])
            freq_pct = float(st["freq_pct"])
            mean_conf = float(st["mean_conf"])
            obj_idf = float(idf.get(obj, 1.0))

            if image_count >= tau:
                score = freq_pct * mean_conf * obj_idf
                scored.append((obj, score))
                total_score += score
                obj_rows[obj] = {
                    "image_count": image_count,
                    "freq_pct": freq_pct,
                    "mean_conf": mean_conf,
                    "idf": obj_idf,
                    "score": float(score),
                }
            else:
                obj_rows[obj] = {
                    "image_count": image_count,
                    "freq_pct": freq_pct,
                    "mean_conf": mean_conf,
                    "idf": obj_idf,
                    "score": 0.0,
                }

        scored.sort(key=lambda x: x[1], reverse=True)

        key_objects: List[str] = []
        if total_score > 0:
            cum = 0.0
            for obj, scv in scored:
                key_objects.append(obj)
                cum += scv
                if cum >= coverage_target * total_score:
                    break

        vkb[scene] = {
            "num_images": N,
            "objects": obj_rows,
            "key_objects": key_objects,
            "selection_rule": {
                "min_support": "tau(s) = max(3, ceil(0.02*N_s))",
                "trimmed_mean_tail": f"t_s = min(0.10, 10/N_s) = {t_tail:.4f} per tail",
                "coverage_target": float(coverage_target),
                "idf": "log((S+1)/(df+1))+1 with scene-relative presence",
            },
        }

    return vkb


# ----------------------------
# MAIN
# ----------------------------
def main():
    # 1) List images by split/scene
    scene2imgs = list_scenes_and_images(DATA_ROOT, SPLIT)
    print(f"\n=== Building VKB from split: '{SPLIT}' | Scenes: {len(scene2imgs)} ===")

    # 2) Load YOLO and detect
    model = YOLO(MODEL_WEIGHTS)
    all_paths = [p for paths in scene2imgs.values() for p in paths]
    print(f"Total images: {len(all_paths)}")

    det_map = predict_paths(model, all_paths, batch_size=INIT_BATCH)  # {image_path: {obj: max_conf}}

    # 3) Aggregate per scene
    scene_stats, all_objects = aggregate_scene_stats(scene2imgs, det_map)
    print(f"Total unique objects detected: {len(all_objects)}")

    # 4) Compute tau and idf across scenes
    tau_by_scene = compute_tau_by_scene(scene_stats)
    idf = compute_idf(scene_stats, all_objects, tau_by_scene)

    # 5 & 6) Score and select key objects (coverage rule)
    vkb = build_clean_vkb(scene_stats, idf, coverage_target=COVERAGE_TARGET)

    # Save
    save_json(OUTPUT_CLEAN_VKB_PATH, vkb)
    print(f"\n💾 VKB saved to: {OUTPUT_CLEAN_VKB_PATH}")
    for scene, sc in vkb.items():
        print(f"• {scene}: {len(sc['objects'])} objects, {len(sc['key_objects'])} key objects")

if __name__ == "__main__":
    main()


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

=== Building VKB from split: 'train' | Scenes: 8 ===


Total images: 4000
Total unique objects detected: 70

💾 VKB saved to: /content/drive/MyDrive/Scene_Classification/clean_vkb_new_from_split.json
• bedroom: 35 objects, 6 key objects
• bus_station: 28 objects, 4 key objects
• highway: 33 objects, 5 key objects
• kitchen: 42 objects, 11 key objects
• living_room: 35 objects, 8 key objects
• office: 46 objects, 12 key objects
• supermarket: 49 objects, 10 key objects
• train_station: 27 objects, 3 key objects


In [ ]:
# ============================================================
# Zero-shot Scene Classification (Simple, VKB-weighted)
# - Uses ALL detected objects per image
# - Scene weights = normalized VKB scores
# - Prediction = argmax scene score (unknown if all zero)
# ============================================================

# If running in Colab:
# !pip install ultralytics --quiet

import os, json, gc
from typing import Dict, List, Any
from collections import Counter

import torch
from ultralytics import YOLO

# ----------------------------
# CONFIG — edit these paths
# ----------------------------
IMAGES_DIR = "/content/drive/MyDrive/Scene_Classification/final_Split2/zero_shot_anon"  # folder of images to classify
CLEAN_VKB_PATH = "/content/drive/MyDrive/Scene_Classification/clean_vkb_new_from_split.json"  # VKB built from train
OUTPUT_JSON = "/content/drive/MyDrive/Scene_Classification/predictions_zero_shot_simple.json"

# YOLO settings (match your training-time inference settings)
MODEL_WEIGHTS = "yolov8l.pt"
IMG_SIZE = 640
CONF_THRES = 0.25
IOU_THRES = 0.6
MAX_DET = 100
INIT_BATCH = 32
MIN_BATCH = 4
DEVICE = 0 if torch.cuda.is_available() else "cpu"
USE_HALF = torch.cuda.is_available()
SUPPORTED_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

torch.backends.cudnn.benchmark = True


# ----------------------------
# I/O helpers
# ----------------------------
def load_json(path: str) -> Any:
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json(path: str, data: Any) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)

def list_images(folder: str) -> List[str]:
    if not os.path.isdir(folder):
        raise RuntimeError(f"Folder not found: {folder}")
    paths = [os.path.join(folder, fn) for fn in sorted(os.listdir(folder))
             if fn.lower().endswith(SUPPORTED_EXTS)]
    if not paths:
        raise RuntimeError(f"No images found in: {folder}")
    return paths


# ----------------------------
# Build scene profiles from VKB
# weights_s(o) = score(o,s) / sum_o' score(o',s)  (only positive scores)
# ----------------------------
def build_scene_profiles_from_vkb(clean_vkb: Dict[str, Any]) -> Dict[str, Dict[str, float]]:
    profiles: Dict[str, Dict[str, float]] = {}
    for scene, block in clean_vkb.items():
        obj_stats = block.get("objects", {}) or {}
        weights = {}
        total = 0.0
        for o, st in obj_stats.items():
            w = float(st.get("score", 0.0))
            if w > 0:
                weights[o] = w
                total += w
        if total > 0:
            for o in list(weights.keys()):
                weights[o] /= total
        profiles[scene] = weights
    return profiles


# ----------------------------
# YOLO detection: unique-per-image max confidence
# Returns: {image_path: {object_name: max_conf}}
# ----------------------------
def detect_unique_max(model: YOLO, paths: List[str], batch_size: int) -> Dict[str, Dict[str, float]]:
    out: Dict[str, Dict[str, float]] = {}
    i, bs = 0, batch_size
    while i < len(paths):
        chunk = paths[i:i+bs]
        try:
            with torch.inference_mode():
                results = model.predict(
                    source=chunk, imgsz=IMG_SIZE, conf=CONF_THRES, iou=IOU_THRES,
                    max_det=MAX_DET, device=DEVICE, half=USE_HALF, verbose=False
                )
            for p, r in zip(chunk, results):
                per_image = {}
                if getattr(r, "boxes", None) is not None and len(r.boxes) > 0:
                    names_map = r.names if getattr(r, "names", None) else model.names
                    for b in r.boxes:
                        cls_id = int(b.cls.item())
                        name = names_map[cls_id]
                        conf = float(b.conf.item())
                        if (name not in per_image) or (conf > per_image[name]):
                            per_image[name] = conf
                out[p] = per_image
            i += bs
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            gc.collect()
            if bs <= MIN_BATCH:
                raise
            bs = max(MIN_BATCH, bs // 2)
        except Exception as e:
            print(f"⚠️ Error near index {i}: {e}")
            if len(chunk) > 1:
                # process first now, requeue the rest
                for q in reversed(chunk[1:]):
                    paths.insert(i + 1, q)
                chunk = [chunk[0]]
                bs = max(MIN_BATCH, 1)
            else:
                out[chunk[0]] = {}
                i += 1
        finally:
            torch.cuda.empty_cache()
    return out


# ----------------------------
# Score image vs all scenes:
# scene_score(s) = Σ_o conf(o) * weights_s(o)
# ----------------------------
def score_image(per_image: Dict[str, float], profiles: Dict[str, Dict[str, float]]) -> Dict[str, float]:
    scores: Dict[str, float] = {}
    for scene, weights in profiles.items():
        s = 0.0
        for o, conf in per_image.items():
            w = weights.get(o, 0.0)
            if w > 0:
                s += w * float(conf)
        scores[scene] = s
    return scores


# ----------------------------
# Decide label: argmax; "unknown" if all scores are exactly 0
# ----------------------------
def decide_label(scores: Dict[str, float]) -> str:
    if not scores:
        return "unknown"
    best_scene, best_score = max(scores.items(), key=lambda kv: kv[1])
    if all(v == 0.0 for v in scores.values()):
        return "unknown"
    return best_scene


# ----------------------------
# MAIN
# ----------------------------
def main():
    # Load VKB and prepare scene profiles
    clean_vkb = load_json(CLEAN_VKB_PATH)
    profiles = build_scene_profiles_from_vkb(clean_vkb)
    scenes = sorted(profiles.keys())
    if not scenes:
        raise RuntimeError("VKB has no scenes.")

    # Collect images
    img_paths = list_images(IMAGES_DIR)
    print(f"\nClassifying {len(img_paths)} images into scenes:\n  {', '.join(scenes)}")

    # Detect
    model = YOLO(MODEL_WEIGHTS)
    det_map = detect_unique_max(model, img_paths, batch_size=INIT_BATCH)

    # Classify
    preds = []
    counts = Counter()
    for p in img_paths:
        per_image = det_map.get(p, {})
        scores = score_image(per_image, profiles)
        label = decide_label(scores)
        preds.append({
            "image_path": p,
            "pred_scene": label,
            "scores": dict(sorted(scores.items(), key=lambda kv: kv[1], reverse=True)),
            "detected_objects": per_image
        })
        counts[label] += 1

    # Summary
    print("\nPrediction counts:")
    for k, v in sorted(counts.items(), key=lambda kv: (-kv[1], kv[0])):
        print(f"  {k:20s} : {v}")
    print(f"Total: {len(preds)}")

    # Save predictions
    save_json(OUTPUT_JSON, {
        "images_dir": IMAGES_DIR,
        "vkb_path": CLEAN_VKB_PATH,
        "predictions": preds
    })
    print(f"\n💾 Saved predictions: {OUTPUT_JSON}")

if __name__ == "__main__":
    main()



Classifying 400 images into scenes:
  bedroom, bus_station, highway, kitchen, living_room, office, supermarket, train_station

Prediction counts:
  supermarket          : 72
  living_room          : 62
  bedroom              : 49
  bus_station          : 45
  train_station        : 45
  unknown              : 43
  kitchen              : 38
  highway              : 26
  office               : 20
Total: 400

💾 Saved predictions: /content/drive/MyDrive/Scene_Classification/predictions_zero_shot_simple.json


In [ ]:
# ==========================================
# EVALUATION — with CSV mapping (flat folder)
# Requires: scikit-learn
# !pip install scikit-learn --quiet
# ==========================================

import os, json, pandas as pd
from sklearn.metrics import confusion_matrix, classification_report

# ---- EDIT THESE PATHS ----
PREDICTIONS_JSON = "/content/drive/MyDrive/Scene_Classification/predictions_zero_shot_simple.json"
MAPPING_CSV      = "/content/drive/MyDrive/Scene_Classification/final_Split2/zero_shot_mapping.csv"

# 1) Load predictions (from your classifier)
with open(PREDICTIONS_JSON, "r", encoding="utf-8") as f:
    preds_blob = json.load(f)
pred_rows = preds_blob["predictions"]

pred_df = pd.DataFrame([{
    "image_path": r["image_path"],
    "filename": os.path.basename(r["image_path"]),
    "pred_scene": r["pred_scene"],
    "scores": r["scores"],
} for r in pred_rows])

# 2) Load mapping CSV and detect columns
map_df = pd.read_csv(MAPPING_CSV)
possible_img_cols = ["image_path","image","path","filepath","file_path","filename","file","anon_name","anon_filename"]
possible_lbl_cols = ["scene","label","true_label","scene_label","gt","ground_truth"]

img_col = next((c for c in possible_img_cols if c in map_df.columns), None)
lbl_col = next((c for c in possible_lbl_cols if c in map_df.columns), None)
if img_col is None or lbl_col is None:
    raise ValueError(f"CSV must include image + label columns. Found: {list(map_df.columns)}")

map_df["__filename__"] = map_df[img_col].apply(lambda x: os.path.basename(str(x)))
pred_df["__filename__"] = pred_df["filename"]

merged = pd.merge(pred_df, map_df, on="__filename__", how="inner")
if merged.empty:
    raise RuntimeError("No matches between predictions and mapping CSV. Check file names/paths.")

# 3) Overall accuracy (count 'unknown' as wrong)
y_true = merged[lbl_col].tolist()
y_pred = merged["pred_scene"].tolist()
acc = (merged["pred_scene"] == merged[lbl_col]).mean()
unknown_rate = (merged["pred_scene"] == "unknown").mean()

print(f"Images evaluated: {len(merged)}")
print(f"Accuracy (unknown counted wrong): {acc*100:.2f}%")
print(f"'unknown' rate: {unknown_rate*100:.2f}%")

# 4) Confusion matrix & classification report (exclude 'unknown')
scenes = sorted(set(y_true))  # expected scene labels
mask_valid = merged["pred_scene"] != "unknown"
cm = confusion_matrix(merged.loc[mask_valid, lbl_col],
                      merged.loc[mask_valid, "pred_scene"],
                      labels=scenes)
print("\nConfusion matrix (valid preds only):")
print(pd.DataFrame(cm, index=scenes, columns=scenes))

print("\nClassification report (valid preds only):")
print(classification_report(merged.loc[mask_valid, lbl_col],
                            merged.loc[mask_valid, "pred_scene"],
                            labels=scenes,
                            zero_division=0))

# 5) Top-1 and Top-2 stats (optional but informative)
# top-1 = same as accuracy excluding 'unknown' denominator differences
# top-2 = whether true scene appears in the 2 highest scores
def top2_contains_true(row):
    items = sorted(row["scores"].items(), key=lambda kv: kv[1], reverse=True)
    top2 = [items[0][0]] + ([items[1][0]] if len(items) > 1 else [])
    return row[lbl_col] in top2

top1 = (merged["pred_scene"] == merged[lbl_col]).mean()
top2 = merged.apply(top2_contains_true, axis=1).mean()
print(f"\nTop-1 accuracy (all): {top1*100:.2f}%")
print(f"Top-2 contains true:  {top2*100:.2f}%")


Images evaluated: 400
Accuracy (unknown counted wrong): 68.50%
'unknown' rate: 10.75%

Confusion matrix (valid preds only):
               bedroom  bus_station  highway  kitchen  living_room  office  \
bedroom             47            0        0        0            1       0   
bus_station          0           45        1        0            0       0   
highway              0            0       23        0            0       0   
kitchen              0            0        0       34            8       1   
living_room          2            0        0        0           41       1   
office               0            0        0        1           12      17   
supermarket          0            0        1        3            0       1   
train_station        0            0        1        0            0       0   

               supermarket  train_station  
bedroom                  1              1  
bus_station              4              0  
highway                  5              2

In [3]:
# ========================== Setup & Imports ===================================
!pip install -q ultralytics scikit-learn pandas

import os, json, glob, random, shutil, csv
from typing import List, Tuple, Dict
from collections import defaultdict
import numpy as np
from tqdm import tqdm
import pandas as pd

from ultralytics import YOLO
from sklearn.metrics import classification_report, confusion_matrix

# ========================== User Config =======================================
DATA_ROOT = "/content/drive/MyDrive/Scene_Classification/Dataset"   # expects DATA_ROOT/scene_name/*.jpg
VKB_JSON  = "/content/drive/MyDrive/Scene_Classification/clean_vkb_new_from_split.json"
OUT_DIR   = "/content/drive/MyDrive/Scene_Classification/subset_50_flat"  # will be created/overwritten

SCENES = ["bedroom","kitchen","office","supermarket",
          "bus_station","train_station","highway","living_room"]
SAMPLES_PER_SCENE = 50
RANDOM_SEED = 1234

# YOLO params (match your final code)
YOLO_MODEL = "yolov8l.pt"
IMG_SIZE   = 640
CONF_THRES = 0.25
IOU_THRES  = 0.60
MAX_DETS   = 100

# ========================== Helpers ===========================================
def list_images(scene_dir: str) -> List[str]:
    exts = ("*.jpg","*.jpeg","*.png","*.bmp","*.webp")
    files = []
    for e in exts:
        files.extend(glob.glob(os.path.join(scene_dir, e)))
    return sorted(files)

def sample_subset_per_scene(data_root: str, scenes: List[str], k: int, seed: int) -> List[Tuple[str,str]]:
    """Return [(img_path, scene_label)] exactly k per scene (random, reproducible)."""
    rng = random.Random(seed)
    sampled = []
    for s in scenes:
        sdir = os.path.join(data_root, s)
        imgs = list_images(sdir)
        if len(imgs) < k:
            raise ValueError(f"Scene '{s}' has only {len(imgs)} images, need {k}.")
        rng.shuffle(imgs)
        sampled.extend([(p, s) for p in imgs[:k]])
    return sampled

def flatten_and_save_mapping(pairs: List[Tuple[str,str]], out_dir: str):
    """Copy images into one flat folder with scene-prefixed filenames.
       Save subset_mapping.json and subset_mapping.csv."""
    if os.path.exists(out_dir):
        shutil.rmtree(out_dir)
    os.makedirs(out_dir, exist_ok=True)

    mapping = []
    counters = {s: 0 for s in SCENES}
    for src, scene in pairs:
        counters[scene] += 1
        ext = os.path.splitext(src)[1].lower()
        fname = f"{scene}_{counters[scene]:03d}{ext}"
        dst = os.path.join(out_dir, fname)
        shutil.copy2(src, dst)
        mapping.append({
            "image_path_flat": dst,
            "image_path_src": src,
            "scene": scene,
            "filename": fname
        })

    # JSON
    with open(os.path.join(out_dir, "subset_mapping.json"), "w") as f:
        json.dump(mapping, f, indent=2)

    # CSV
    with open(os.path.join(out_dir, "subset_mapping.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["filename","scene","image_path_flat","image_path_src"])
        w.writeheader()
        for row in mapping:
            w.writerow({
                "filename": row["filename"],
                "scene": row["scene"],
                "image_path_flat": row["image_path_flat"],
                "image_path_src": row["image_path_src"],
            })

    print(f"✅ Flattened {len(mapping)} files to: {out_dir}")
    print(f"✅ Saved mapping: subset_mapping.json and subset_mapping.csv")
    return mapping

def load_vkb(vkb_path: str) -> Dict:
    with open(vkb_path, "r") as f:
        return json.load(f)

def build_scene_weights_from_vkb(vkb: Dict, use_key_objects: bool, weight_mode: str, l1_normalise: bool) -> Dict[str, Dict[str,float]]:
    """
    weight_mode in {"score","no_idf","freq_only","conf_only","uniform"}
    use_key_objects: True => only VKB['key_objects'] ; False => all objects
    l1_normalise: per-scene L1 normalisation of weights
    """
    scene_w = {}
    for scene, payload in vkb.items():
        objs = payload.get("objects", {})
        chosen = payload.get("key_objects", []) if use_key_objects else list(objs.keys())
        weights = {}
        for o in chosen:
            stats = objs.get(o, {})
            if weight_mode == "score":
                val = float(stats.get("score", 0.0))
            elif weight_mode == "no_idf":
                val = float(stats.get("freq_pct", 0.0)) * float(stats.get("mean_conf", 0.0))
            elif weight_mode == "freq_only":
                val = float(stats.get("freq_pct", 0.0))
            elif weight_mode == "conf_only":
                val = float(stats.get("mean_conf", 0.0))
            elif weight_mode == "uniform":
                val = 1.0
            else:
                raise ValueError(f"Unknown weight_mode: {weight_mode}")
            weights[o] = max(0.0, val)
        if l1_normalise:
            s = sum(weights.values())
            if s > 0:
                weights = {o: v/s for o, v in weights.items()}
        scene_w[scene] = weights
    return scene_w

def detect_objects_yolo(model: YOLO, img_path: str, max_per_class: bool=True) -> Dict[str,float]:
    """
    Run YOLO and return {object_name: aggregated_conf}.
    If max_per_class=True: keep only the highest confidence per class; else sum confidences.
    """
    out = model.predict(img_path, imgsz=IMG_SIZE, conf=CONF_THRES, iou=IOU_THRES,
                        max_det=MAX_DETS, verbose=False)[0]
    per_class = {}
    for b in out.boxes:
        name = model.names[int(b.cls)]
        conf = float(b.conf)
        if name not in per_class:
            per_class[name] = conf
        else:
            per_class[name] = max(per_class[name], conf) if max_per_class else (per_class[name] + conf)
    return per_class

def score_image(objects_conf: Dict[str,float], scene_weights: Dict[str,Dict[str,float]]) -> Dict[str,float]:
    """S(I,s) = sum_o conf(o) * w(o,s) over intersection."""
    scores = {}
    for s, wmap in scene_weights.items():
        sc = 0.0
        for o, conf in objects_conf.items():
            w = wmap.get(o, 0.0)
            if w > 0 and conf > 0:
                sc += conf * w
        scores[s] = sc
    return scores

def evaluate_run(y_true: List[str], y_pred: List[str], scores_list: List[Dict[str,float]]):
    # Overall accuracy with "unknown" counted wrong
    correct = sum(yt == yp for yt, yp in zip(y_true, y_pred))
    acc = 100.0 * correct / len(y_true)
    # Unknown rate
    unk = sum(yp == "unknown" for yp in y_pred)
    unk_rate = 100.0 * unk / len(y_pred)
    # Top-2 containment
    top2_hits = 0
    for yt, sc in zip(y_true, scores_list):
        top2 = sorted(sc.items(), key=lambda x: x[1], reverse=True)[:2]
        if any(s == yt for s, _ in top2):
            top2_hits += 1
    top2_cont = 100.0 * top2_hits / len(y_true)

    print(f"Images evaluated: {len(y_true)}")
    print(f"Accuracy (unknown counted wrong): {acc:.2f}%")
    print(f"'unknown' rate: {unk_rate:.2f}%")
    print(f"Top-2 containment: {top2_cont:.2f}%\n")

    # Valid predictions only
    y_true_valid = [yt for yt, yp in zip(y_true, y_pred) if yp != "unknown"]
    y_pred_valid = [yp for yp in y_pred if yp != "unknown"]
    if len(y_true_valid) > 0:
        print("Classification report (valid preds only):")
        print(classification_report(y_true_valid, y_pred_valid, labels=SCENES, zero_division=0))
        print("Confusion matrix (valid preds only):")
        print(confusion_matrix(y_true_valid, y_pred_valid, labels=SCENES))
    else:
        print("No valid predictions (all unknown).")

# ========================== Ablations (VKB unchanged) =========================
ABLATIONS = [
    {
        "name": "final_like_weights_key_norm_max_with_abstain",
        "weight_mode": "score",
        "use_key_objects": True,
        "l1_normalise": True,
        "max_per_class": True,
        "force_prediction": False
    },
    {
        "name": "no_idf_keep_key_norm_max_with_abstain",
        "weight_mode": "no_idf",
        "use_key_objects": True,
        "l1_normalise": True,
        "max_per_class": True,
        "force_prediction": False
    },
    {
        "name": "no_normalisation_keep_key_score_max_with_abstain",
        "weight_mode": "score",
        "use_key_objects": True,
        "l1_normalise": False,
        "max_per_class": True,
        "force_prediction": False
    },
    {
        "name": "use_all_objects_score_norm_max_with_abstain",
        "weight_mode": "score",
        "use_key_objects": False,
        "l1_normalise": True,
        "max_per_class": True,
        "force_prediction": False
    },
    {
        "name": "uniform_weights_key_norm_max_with_abstain",
        "weight_mode": "uniform",
        "use_key_objects": True,
        "l1_normalise": True,
        "max_per_class": True,
        "force_prediction": False
    },
    {
        "name": "final_like_but_sum_per_class_no_max_with_abstain",
        "weight_mode": "score",
        "use_key_objects": True,
        "l1_normalise": True,
        "max_per_class": False,
        "force_prediction": False
    },
    {
        "name": "final_like_force_prediction_no_abstain",
        "weight_mode": "score",
        "use_key_objects": True,
        "l1_normalise": True,
        "max_per_class": True,
        "force_prediction": True
    },
    {
        "name": "freq_only_key_norm_max_with_abstain",
        "weight_mode": "freq_only",
        "use_key_objects": True,
        "l1_normalise": True,
        "max_per_class": True,
        "force_prediction": False
    },
    {
        "name": "conf_only_key_norm_max_with_abstain",
        "weight_mode": "conf_only",
        "use_key_objects": True,
        "l1_normalise": True,
        "max_per_class": True,
        "force_prediction": False
    },
]

# ========================== Main =============================================
def main():
    random.seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)

    # 1) Sample & flatten subset (50 per scene) + save mapping
    sampled = sample_subset_per_scene(DATA_ROOT, SCENES, SAMPLES_PER_SCENE, RANDOM_SEED)
    mapping = flatten_and_save_mapping(sampled, OUT_DIR)

    # 2) Load VKB (unchanged)
    vkb = load_vkb(VKB_JSON)

    # 3) Load YOLO once
    model = YOLO(YOLO_MODEL)

    # 4) YOLO detections ONCE for the flat subset (cache max-per-class)
    print("Running YOLO detections once on the flattened subset...")
    det_cache = []
    for row in tqdm(mapping):
        img = row["image_path_flat"]
        gt  = row["scene"]
        per_class_max = detect_objects_yolo(model, img, max_per_class=True)
        det_cache.append({"img": img, "gt": gt, "per_class_max": per_class_max})

    # 5) Run ablations using the SAME subset & cached detections
    for abl in ABLATIONS:
        print("\n" + "="*80)
        print(f"Ablation: {abl['name']}")
        print("="*80)

        # Build scene weights from VKB (unchanged VKB file)
        scene_w = build_scene_weights_from_vkb(
            vkb=vkb,
            use_key_objects=abl["use_key_objects"],
            weight_mode=abl["weight_mode"],
            l1_normalise=abl["l1_normalise"]
        )

        y_true, y_pred, scores_list = [], [], []
        detected_list = []  # keep the actual per-image objects used

        for entry in det_cache:
            img_path = entry["img"]
            gt = entry["gt"]

            if abl["max_per_class"]:
                objs_conf = entry["per_class_max"]
            else:
                # re-run YOLO for sum aggregation (only for this ablation)
                objs_conf = detect_objects_yolo(model, img_path, max_per_class=False)

            sc = score_image(objs_conf, scene_w)
            scores_list.append(sc)
            detected_list.append(objs_conf)

            if abl["force_prediction"]:
                pred = max(sc.items(), key=lambda x: x[1])[0]
            else:
                max_s, max_v = max(sc.items(), key=lambda x: x[1])
                pred = max_s if max_v > 0 else "unknown"

            y_true.append(gt)
            y_pred.append(pred)

        # Evaluate
        evaluate_run(y_true, y_pred, scores_list)

        # 6) Save predictions per ablation (for interpretability & appendix)
        save_data = []
        for row, pred, sc, objs in zip(mapping, y_pred, scores_list, detected_list):
            save_data.append({
                "image_path": row["image_path_flat"],
                "ground_truth": row["scene"],
                "pred_scene": pred,
                "scores": sc,                 # per-scene score vector
                "detected_objects": objs      # aggregated confidences (max or sum per ablation)
            })
        out_json = os.path.join(OUT_DIR, f"predictions_{abl['name']}.json")
        with open(out_json, "w") as f:
            json.dump(save_data, f, indent=2)
        print(f"💾 Saved: {out_json}")

if __name__ == "__main__":
    main()

✅ Flattened 400 files to: /content/drive/MyDrive/Scene_Classification/subset_50_flat
✅ Saved mapping: subset_mapping.json and subset_mapping.csv
Running YOLO detections once on the flattened subset...


100%|██████████| 400/400 [19:43<00:00,  2.96s/it]



Ablation: final_like_weights_key_norm_max_with_abstain
Images evaluated: 400
Accuracy (unknown counted wrong): 68.50%
'unknown' rate: 12.50%
Top-2 containment: 81.00%

Classification report (valid preds only):
               precision    recall  f1-score   support

      bedroom       1.00      0.85      0.92        48
      kitchen       0.90      0.77      0.83        48
       office       0.93      0.29      0.44        45
  supermarket       0.37      0.88      0.52        32
  bus_station       0.98      0.92      0.95        50
train_station       0.96      0.92      0.94        50
      highway       0.92      0.79      0.85        29
  living_room       0.68      0.83      0.75        48

     accuracy                           0.78       350
    macro avg       0.84      0.78      0.78       350
 weighted avg       0.86      0.78      0.79       350

Confusion matrix (valid preds only):
[[41  1  0  2  0  0  0  4]
 [ 0 37  0  5  0  0  0  6]
 [ 0  2 13 22  0  0  0  8]
 [ 0  1 